# TableTrace GPU 실험: EXAONE LoRA → vLLM / SGLang → 멀티 GPU

**이 노트북은 Colab 또는 Kaggle에서 실행합니다. 작성자가 실행해 보지 못했습니다**(개발 컨테이너에 GPU가 없고 Hugging Face가 막혀 있음). 아래 명령은 저장소의 스크립트를 부르는 것이고, 스크립트의 학습 루프·엔진 연결은 CPU에서 작은 모델과 모의 서버로 시험했습니다(`tests/test_lora_sft_smoke.py`, `tests/test_engines.py`). 실제 EXAONE·vLLM·SGLang·NCCL 실행에서 생기는 문제는 여기서 처음 드러날 수 있습니다.

| 하고 싶은 것 | 필요한 것 | Colab 무료(T4 1장) | Kaggle(T4 2장) |
|---|---|---|---|
| 1. EXAONE LoRA 튜닝 | GPU 1장 | 가능 | 가능 |
| 2. vLLM 서빙·평가·벤치 | GPU 1장 | 가능 | 가능 |
| 3. SGLang 서빙·평가·벤치 | GPU 1장 | 가능(시도) | 가능(시도) |
| 4. 텐서 병렬 TP=2 | **GPU 2장** | 불가 | 가능 |
| 5. 멀티 GPU 모델 병렬(mT5 2단계 파이프라인) | **GPU 2장** | 불가 | 가능 |

T4는 bfloat16을 지원하지 않습니다(vLLM 0.32는 T4에서 float16·float32만 목록에 올림). 그래서 아래는 fp16입니다. 라이선스: EXAONE은 비상업 연구용(EXAONE AI Model License 1.2-NC)이고 Hugging Face에서 동의가 필요할 수 있습니다. 토큰은 Colab **Secrets**(`HF_TOKEN`)에 넣으세요.

### Kaggle에서 하는 경우 (GPU 2장이 필요한 4번 절까지 하려면 Kaggle)

1. kaggle.com 계정에서 **휴대폰 인증**을 마쳐야 GPU와 인터넷을 켤 수 있습니다.
2. Notebooks → *New Notebook* → File → *Import Notebook*에서 이 `.ipynb`를 올리거나 GitHub 주소를 붙입니다.
3. 오른쪽 **Session options**: Accelerator = *GPU T4 x2*, Internet = *On*.
4. Add-ons → **Secrets**에 `HF_TOKEN`을 추가하고 이 노트북에 연결합니다.
5. 위에서 아래로 실행합니다. 무료 GPU는 주 30시간, 세션은 최대 12시간입니다. 결과는 마지막 셀이 `/kaggle/working`에 만드는 `gpu_results.tgz`입니다(Output 탭에서 받기).
6. 디스크: `/kaggle/working`은 약 20GB로 제한되어 있어 vLLM·SGLang 가상환경은 `/tmp`에 만듭니다(아래 셀). 그래도 모자라면 vLLM을 끝낸 뒤 `rm -rf /tmp/venv-vllm`으로 지우고 SGLang을 설치하세요.
7. 1~3번 절은 GPU 한 장만 씁니다. 4번 절에서 두 장을 씁니다.

In [ ]:
!nvidia-smi
import torch
print("GPU 수:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("compute capability:", torch.cuda.get_device_capability(0), "(7.5 = T4: bf16 없음 → fp16)")

## 0. 준비 — 저장소, 의존성, 토큰

In [ ]:
import os, subprocess
try:                                   # Colab Secrets
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    try:                               # Kaggle Secrets
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("HF_TOKEN을 Secrets에 넣지 않았습니다. 공개 모델만 받을 수 있습니다.")

REPO = "https://github.com/sokldjs554/tabletrace"      # 비공개 저장소면 https://<토큰>@github.com/... 로 바꾸세요
ROOT = "/content" if os.path.isdir("/content") else "/kaggle/working"      # Colab / Kaggle
VENV = ROOT if ROOT == "/content" else "/tmp"                                  # Kaggle: working dir is only ~20GB
if not os.path.isdir(f"{ROOT}/tabletrace"):
    subprocess.run(["git", "clone", "-q", REPO, f"{ROOT}/tabletrace"], check=True)
os.chdir(f"{ROOT}/tabletrace")
!pip -q install -e ".[dev]" peft accelerate sentencepiece
!pip -q uninstall -y torchao      # Colab의 torchao 0.10.0은 peft가 요구하는 0.16 이상보다 낮아 LoRA 주입에서 ImportError. 이 프로젝트는 torchao를 쓰지 않음

## 1. EXAONE LoRA 튜닝 (같은 데이터, 같은 프로그램 문법)

- `LGAI-EXAONE/EXAONE-4.0-1.2B`는 transformers가 기본 지원하고 T4 메모리에 여유가 있습니다. `EXAONE-3.5-2.4B-Instruct`를 쓰려면 `TRUST="--trust-remote-code"`를 켜고, transformers 버전 충돌이 나면 4.0-1.2B로 돌아가세요.
- 학습 시간은 **실측이 없습니다.** 3,654개 예제 1 epoch은 T4에서 1시간 안팎일 수 있다는 어림값이라, 먼저 `LIMIT=300`으로 몇 분짜리 시험을 해서 손실이 내려가는지 확인하세요.
- 본문 예제(`text2_*`)는 KorQuAD 파생이라 저장소에 없습니다. 여기서는 표 조회·계산·보류 예제만 씁니다.

In [ ]:
MODEL = "LGAI-EXAONE/EXAONE-4.0-1.2B"
TRUST = ""            # EXAONE-3.5 라면 "--trust-remote-code"
LIMIT = 300           # 시험: 300, 본 학습: 0 (전부)
!CUDA_VISIBLE_DEVICES=0 python scripts/train/exaone_lora_sft.py --model {MODEL} {TRUST} --dtype auto --grad-ckpt --merge \
    --train data/ko/train.jsonl data/ko/synthetic_train.jsonl --corpus data/ko/documents.jsonl \
    --epochs 1 --bs 2 --grad-accum 8 --limit {LIMIT} --out experiments/exaone_lora

## 2. vLLM — 서빙, 같은 검증기로 평가, 벤치

vLLM과 SGLang은 torch 버전을 다르게 고정하므로 **가상환경을 따로** 만듭니다(Colab 기본 torch를 건드리지 않음). 설치에 몇 분 걸립니다.

In [ ]:
!pip -q install uv
!uv venv {VENV}/venv-vllm -q
!uv pip install -q --python {VENV}/venv-vllm/bin/python vllm

In [ ]:
import os, subprocess, time, requests

def serve(script, venv, port, log, **env):
    e = dict(os.environ, PORT=str(port), PATH=f"{venv}/bin:" + os.environ["PATH"], **{k: str(v) for k, v in env.items()})
    proc = subprocess.Popen(["bash", script], env=e, stdout=open(log, "w"), stderr=subprocess.STDOUT)
    for _ in range(240):
        try:
            requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=2).raise_for_status()
            return proc
        except Exception:
            if proc.poll() is not None:
                break
            time.sleep(5)
    print(open(log).read()[-4000:])
    raise RuntimeError("서버가 뜨지 않았습니다. 위 로그를 보세요(메모리 부족이면 MAX_LEN을 줄이거나 EXTRA_ARGS='--enforce-eager').")

MERGED = "experiments/exaone_lora/merged"
vllm_proc = serve("scripts/serving/vllm_tp2.sh", f"{VENV}/venv-vllm", 8001, f"{ROOT}/vllm.log",
                  MODEL=MERGED, NUM_GPUS=1, DTYPE="half", MAX_LEN=2048)

In [ ]:
# 학습한 EXAONE 플래너를 같은 검증기·같은 지표로 평가 (mT5와 비교 가능한 표가 나옴)
!python -m tabletrace.eval.run --planner engine --engine-url http://127.0.0.1:8001 --engine-model planner \
    --tokenizer {MERGED} --max-in 672 --concurrency 16 --corpus data/ko/documents.jsonl \
    --data data/ko/test.jsonl data/ko/synthetic_test.jsonl --out experiments/eval/exaone_vllm_ko_test --no-entity-check
!python -m tabletrace.eval.run --planner engine --replay-from experiments/eval/exaone_vllm_ko_test --corpus data/ko/documents.jsonl \
    --data data/ko/test.jsonl data/ko/synthetic_test.jsonl --out experiments/eval/exaone_vllm_ko_test_ec
for c in (1, 8, 32):
    !python -m tabletrace.bench.engine_bench --url http://127.0.0.1:8001 --model planner --n 64 --concurrency {c} --tokenizer {MERGED} --out experiments/bench/engine_vllm_c{c}.json
vllm_proc.terminate(); vllm_proc.wait()

## 3. SGLang — 같은 모델, 같은 평가 (별도 가상환경)

In [ ]:
!uv venv {VENV}/venv-sglang -q
!uv pip install --python {VENV}/venv-sglang/bin/python "sglang[all]" 2>&1 | tail -n 15
sg_proc = serve("scripts/serving/sglang.sh", f"{VENV}/venv-sglang", 8002, f"{ROOT}/sglang.log",
                MODEL=MERGED, NUM_GPUS=1, DTYPE="half", ATTN_BACKEND="triton")

In [ ]:
# SGLang 서버가 뜨지 않으면 이 셀의 출력 전체를 복사해서 알려 주세요
!nvidia-smi | head -n 4
!{VENV}/venv-sglang/bin/python -c "import sys, torch; print(sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.version.cuda, 'gpu available', torch.cuda.is_available())"
!uv pip list --python {VENV}/venv-sglang/bin/python 2>/dev/null | grep -i -E "^(sglang|sgl-kernel|flashinfer|torch |triton|transformers)"
!tail -n 60 {ROOT}/sglang.log

In [ ]:
!python -m tabletrace.eval.run --planner engine --engine-url http://127.0.0.1:8002 --engine-model planner \
    --tokenizer {MERGED} --max-in 672 --concurrency 16 --corpus data/ko/documents.jsonl \
    --data data/ko/test.jsonl data/ko/synthetic_test.jsonl --out experiments/eval/exaone_sglang_ko_test --no-entity-check
for c in (1, 8, 32):
    !python -m tabletrace.bench.engine_bench --url http://127.0.0.1:8002 --model planner --n 64 --concurrency {c} --tokenizer {MERGED} --out experiments/bench/engine_sglang_c{c}.json
sg_proc.terminate(); sg_proc.wait()

## 4. 멀티 GPU — **Kaggle(T4 ×2) 또는 GPU 2장 이상에서만**

Colab 한 장으로는 아래 두 가지를 할 수 없습니다. Kaggle 노트북 설정에서 Accelerator를 *GPU T4 x2*로, Internet을 켜세요.

(a) **vLLM 텐서 병렬**: 바로 아래 셀입니다. 같은 모델을 GPU 두 장에 나눠 올리고 평가·벤치를 다시 합니다.

(b) **모델 병렬(2단계 파이프라인)**: mT5를 인코더(GPU0)와 디코더+lm_head(GPU1)로 나눠 NCCL로 주고받습니다. 튜닝한 가중치는 git에 없어서 여기서는 **원본 `google/mt5-small`**로 돌립니다. 이 실험이 보는 것은 분할·전송·메모리·"1장과 2장의 출력이 같은가"이지 답변 품질이 아닙니다.

In [ ]:
# (a) vLLM 텐서 병렬 TP=2 — 2번 절(서버를 띄우는 serve 함수와 vLLM 가상환경)을 먼저 실행했어야 합니다.
EXTRA = ""     # 서버가 멈추거나 NCCL 오류가 나면 "--enforce-eager --disable-custom-all-reduce" 로 바꿔 다시 실행
tp2_proc = serve("scripts/serving/vllm_tp2.sh", f"{VENV}/venv-vllm", 8003, f"{ROOT}/vllm_tp2.log",
                 MODEL=MERGED, NUM_GPUS=2, DTYPE="half", MAX_LEN=2048, EXTRA_ARGS=EXTRA)
!nvidia-smi --query-gpu=index,memory.used --format=csv
!python -m tabletrace.eval.run --planner engine --engine-url http://127.0.0.1:8003 --engine-model planner \
    --tokenizer {MERGED} --max-in 672 --concurrency 16 --corpus data/ko/documents.jsonl \
    --data data/ko/test.jsonl data/ko/synthetic_test.jsonl --out experiments/eval/exaone_vllm_tp2_ko_test --no-entity-check
for c in (1, 8, 32):
    !python -m tabletrace.bench.engine_bench --url http://127.0.0.1:8003 --model planner --n 64 --concurrency {c} --tokenizer {MERGED} --out experiments/bench/engine_vllm_tp2_c{c}.json
tp2_proc.terminate(); tp2_proc.wait()

In [ ]:
!python -m tabletrace.bench.pipeline_parallel --model google/mt5-small --single --backend nccl --n 64 --bs 8 --out experiments/bench/pp_single_gpu.json
!torchrun --nproc_per_node=2 -m tabletrace.bench.pipeline_parallel --model google/mt5-small --backend nccl --n 64 --bs 8 --out experiments/bench/pp_2stage_gpu.json

## 5. 결과 돌려주기

`experiments/eval/exaone_*`, `experiments/bench/engine_*`, `experiments/bench/pp_*_gpu.json`, `experiments/exaone_lora/train_info.json`이 산출물입니다. 압축해서 받거나(아래 셀) 저장소의 새 브랜치에 푸시한 뒤 알려 주세요. 그러면 README·보고서의 "구현 후 미실행"을 실측 값으로 바꾸고 `check_numbers`로 검증합니다.

In [ ]:
!tar czf {ROOT}/gpu_results.tgz experiments/eval/exaone_* experiments/bench/engine_* experiments/bench/pp_*_gpu.json experiments/exaone_lora/train_info.json 2>/dev/null
try:
    from google.colab import files; files.download(f"{ROOT}/gpu_results.tgz")
except Exception:
    print(f"{ROOT}/gpu_results.tgz 를 내려받으세요 (Kaggle은 Output 탭).")